# DeepEcoHAB Main Menu Launcher

Run the first code cell (`%gui qt`), then run the launcher cell.


In [1]:
%gui qt

In [ ]:
# DeepEcoHAB Main Menu Notebook Launcher
# Run this cell after running the `%gui qt` cell.

import os
import sys
import json
import time
import shutil
import traceback
import subprocess
from pathlib import Path
from datetime import datetime


from PySide6.QtCore import Qt, QSize
from PySide6.QtGui import QPixmap, QFont, QColor, QPalette
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, QGridLayout,
    QLabel, QPushButton, QMessageBox, QTextEdit, QDialog, QFileDialog,
    QFrame, QSizePolicy
)

DEPS_FOLDER_NAME = "DeepEcoHAB_dependencies"
CONFIG_FILENAME = "main_menu_config.json"
MANUAL_PACKAGE_DIR = ""  # optional: paste full package path here if auto-detection fails


def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def candidate_roots():
    roots = []
    if MANUAL_PACKAGE_DIR.strip():
        roots.append(Path(MANUAL_PACKAGE_DIR.strip()).expanduser())
    try:
        roots.append(Path.cwd())
    except Exception:
        pass
    if "__file__" in globals():
        try:
            roots.append(Path(__file__).resolve().parent)
        except Exception:
            pass
    # VS Code often uses the workspace folder as cwd; also check parents.
    expanded = []
    for r in roots:
        try:
            r = r.resolve()
        except Exception:
            pass
        expanded.append(r)
        expanded.extend(list(r.parents)[:4])
    # Remove duplicates while preserving order.
    out = []
    seen = set()
    for r in expanded:
        k = str(r).lower()
        if k not in seen:
            out.append(r)
            seen.add(k)
    return out


def find_dependencies_folder() -> Path | None:
    for root in candidate_roots():
        dep = root / DEPS_FOLDER_NAME
        if dep.exists() and dep.is_dir():
            return dep
    # Gentle shallow search from cwd and its first parent.
    for root in candidate_roots()[:3]:
        try:
            for dep in root.glob(f"*/{DEPS_FOLDER_NAME}"):
                if dep.exists() and dep.is_dir():
                    return dep
        except Exception:
            pass
    return None


def load_config(deps_dir: Path) -> dict:
    cfg_path = deps_dir / CONFIG_FILENAME
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing config file:\n{cfg_path}")
    return json.loads(cfg_path.read_text(encoding="utf-8"))


def ensure_dirs(deps_dir: Path):
    for name in ["logs", "_launcher_cache"]:
        (deps_dir / name).mkdir(parents=True, exist_ok=True)


def append_log(deps_dir: Path, msg: str):
    ensure_dirs(deps_dir)
    log_path = deps_dir / "logs" / "main_menu_launcher.log"
    with open(log_path, "a", encoding="utf-8", buffering=1) as f:
        f.write(f"[{now_iso()}] {msg}\n")


def strip_jupyter_magic(source: str) -> str:
    out = []
    for line in source.splitlines(True):
        stripped = line.lstrip()
        # Remove IPython magic/shell lines when converting notebook to standalone .py.
        if stripped.startswith("%") or stripped.startswith("!"):
            out.append("# stripped notebook-only line: " + line)
            continue
        out.append(line)
    return "".join(out)


def notebook_to_python(ipynb_path: Path, py_path: Path) -> Path:
    nb = json.loads(ipynb_path.read_text(encoding="utf-8"))
    parts = [
        "# Auto-generated by DeepEcoHAB Main Menu Launcher\n",
        f"# Source notebook: {ipynb_path}\n",
        f"# Generated: {now_iso()}\n\n",
    ]
    for idx, cell in enumerate(nb.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if not src.strip():
            continue
        parts.append(f"\n# ---- notebook code cell {idx} ----\n")
        parts.append(strip_jupyter_magic(src))
        if not parts[-1].endswith("\n"):
            parts.append("\n")
    py_path.parent.mkdir(parents=True, exist_ok=True)
    py_path.write_text("".join(parts), encoding="utf-8")
    return py_path


def resolve_target(deps_dir: Path, target: str) -> Path:
    p = Path(target)
    if not p.is_absolute():
        p = deps_dir / p
    return p.resolve()


def prepare_launch_target(deps_dir: Path, tool: dict) -> Path:
    target = resolve_target(deps_dir, tool.get("target", ""))
    if not target.exists():
        raise FileNotFoundError(f"Tool target not found:\n{target}")
    if target.suffix.lower() == ".ipynb":
        cache = deps_dir / "_launcher_cache"
        safe_id = "".join(ch if ch.isalnum() or ch in "_-" else "_" for ch in tool.get("id", target.stem))
        py_path = cache / f"{safe_id}_launched.py"
        return notebook_to_python(target, py_path)
    return target


def launch_tool(deps_dir: Path, tool: dict) -> subprocess.Popen:
    script_path = prepare_launch_target(deps_dir, tool)
    log_dir = deps_dir / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    tool_id = tool.get("id", script_path.stem)
    tool_log = log_dir / f"{tool_id}_launch.log"
    env = os.environ.copy()
    env["DEEPECOHAB_LAUNCHED_FROM_MENU"] = "1"
    env["DEEPECOHAB_DEPENDENCIES_DIR"] = str(deps_dir)
    env["PYTHONUTF8"] = "1"
    append_log(deps_dir, f"Launching {tool.get('title', tool_id)} using {sys.executable}: {script_path}")
    log_handle = open(tool_log, "a", encoding="utf-8", buffering=1)
    log_handle.write(f"\n\n----- Launch {tool.get('title', tool_id)} at {now_iso()} -----\n")
    log_handle.write(f"Python: {sys.executable}\n")
    log_handle.write(f"Script: {script_path}\n")
    log_handle.flush()
    proc = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=str(script_path.parent),
        env=env,
        stdout=log_handle,
        stderr=log_handle,
    )
    # Keep log handle alive by storing it on process object.
    proc._deepecohab_log_handle = log_handle
    return proc


class HelpDialog(QDialog):
    def __init__(self, help_text: str, parent=None):
        super().__init__(parent)
        self.setWindowTitle("DeepEcoHAB Help - PC preparation")
        self.resize(980, 760)
        layout = QVBoxLayout(self)
        title = QLabel("DeepEcoHAB PC preparation / usage help")
        title.setStyleSheet("font-size: 22px; font-weight: bold;")
        layout.addWidget(title)
        text = QTextEdit()
        text.setReadOnly(True)
        text.setPlainText(help_text)
        layout.addWidget(text, 1)
        close = QPushButton("Close")
        close.clicked.connect(self.close)
        layout.addWidget(close, 0, Qt.AlignRight)


class ToolCard(QFrame):
    def __init__(self, deps_dir: Path, tool: dict, launch_callback, parent=None):
        super().__init__(parent)
        self.deps_dir = deps_dir
        self.tool = tool
        self.launch_callback = launch_callback
        self.setFrameShape(QFrame.StyledPanel)
        self.setObjectName("ToolCard")
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        layout = QVBoxLayout(self)
        layout.setContentsMargins(16, 16, 16, 16)
        layout.setSpacing(10)

        img = QLabel()
        img.setAlignment(Qt.AlignCenter)
        img.setMinimumHeight(190)
        pix = self.load_pixmap(tool.get("image", ""))
        if pix is not None:
            img.setPixmap(pix.scaled(240, 180, Qt.KeepAspectRatio, Qt.SmoothTransformation))
        else:
            img.setText("No image")
        layout.addWidget(img)

        title = QLabel(tool.get("title", "Untitled tool"))
        title.setAlignment(Qt.AlignCenter)
        title.setWordWrap(True)
        title.setStyleSheet("font-size: 20px; font-weight: 700; color: #e6e6e6;")
        layout.addWidget(title)

        desc = QLabel(tool.get("description", ""))
        desc.setAlignment(Qt.AlignCenter)
        desc.setWordWrap(True)
        desc.setStyleSheet("font-size: 12px; color: #b8c2d0;")
        layout.addWidget(desc)

        layout.addStretch(1)
        btn = QPushButton("Launch")
        btn.setMinimumHeight(46)
        btn.clicked.connect(lambda: self.launch_callback(tool))
        btn.setEnabled(bool(tool.get("enabled", True)))
        layout.addWidget(btn)

    def load_pixmap(self, image_rel: str):
        if not image_rel:
            return None
        p = Path(image_rel)
        if not p.is_absolute():
            p = self.deps_dir / p
        if p.exists():
            pix = QPixmap(str(p))
            if not pix.isNull():
                return pix
        return None


class MainMenuWindow(QWidget):
    def __init__(self, deps_dir: Path, config: dict):
        super().__init__()
        self.deps_dir = deps_dir
        self.config = config
        self.processes = []
        self.setWindowTitle(config.get("app_title", "DeepEcoHAB Main Menu"))
        self.resize(1180, 860)
        self.apply_style()

        root = QVBoxLayout(self)
        root.setContentsMargins(22, 18, 22, 18)
        root.setSpacing(14)

        header = QHBoxLayout()
        logo = QLabel()
        logo_path = deps_dir / "assets" / "app_logo.jpg"
        if logo_path.exists():
            pix = QPixmap(str(logo_path))
            if not pix.isNull():
                logo.setPixmap(pix.scaled(86, 86, Qt.KeepAspectRatio, Qt.SmoothTransformation))
        header.addWidget(logo)
        head_text = QVBoxLayout()
        title = QLabel(config.get("app_title", "DeepEcoHAB Main Menu"))
        title.setStyleSheet("font-size: 34px; font-weight: 800;")
        subtitle = QLabel(config.get("app_subtitle", ""))
        subtitle.setStyleSheet("font-size: 15px; color: #b8c2d0;")
        head_text.addWidget(title)
        head_text.addWidget(subtitle)
        header.addLayout(head_text, 1)
        root.addLayout(header)

        grid = QGridLayout()
        grid.setSpacing(16)
        tools = config.get("tools", [])
        # 2 x 2 grid by default; additional tools continue onto new rows.
        for i, tool in enumerate(tools):
            card = ToolCard(deps_dir, tool, self.on_launch_tool)
            row = i // 2
            col = i % 2
            grid.addWidget(card, row, col)
        grid.setColumnStretch(0, 1)
        grid.setColumnStretch(1, 1)
        root.addLayout(grid, 1)

        bottom = QHBoxLayout()
        contact = QLabel(config.get("contact_text", ""))
        contact.setWordWrap(True)
        contact.setStyleSheet("font-size: 12px; color: #94a3b8;")
        bottom.addWidget(contact, 1)
        self.status = QLabel(f"Python kernel: {sys.executable}")
        self.status.setStyleSheet("font-size: 11px; color: #94a3b8;")
        bottom.addWidget(self.status, 2)
        help_btn = QPushButton("HELP")
        help_btn.setMinimumWidth(120)
        help_btn.clicked.connect(self.show_help)
        bottom.addWidget(help_btn, 0)
        root.addLayout(bottom)

    def apply_style(self):
        """DeepEcoHAB dark theme, matched to the New RFID DAQ GUI."""
        app = QApplication.instance()
        if app is not None:
            try:
                app.setStyle("Fusion")
            except Exception:
                pass
            pal = QPalette()
            pal.setColor(QPalette.Window, QColor("#0f1115"))
            pal.setColor(QPalette.WindowText, QColor("#e6e6e6"))
            pal.setColor(QPalette.Base, QColor("#12151c"))
            pal.setColor(QPalette.AlternateBase, QColor("#0f1115"))
            pal.setColor(QPalette.ToolTipBase, QColor("#0f1115"))
            pal.setColor(QPalette.ToolTipText, QColor("#e6e6e6"))
            pal.setColor(QPalette.Text, QColor("#e6e6e6"))
            pal.setColor(QPalette.Button, QColor("#151a23"))
            pal.setColor(QPalette.ButtonText, QColor("#e6e6e6"))
            pal.setColor(QPalette.BrightText, QColor("#ffffff"))
            pal.setColor(QPalette.Highlight, QColor("#2d6cdf"))
            pal.setColor(QPalette.HighlightedText, QColor("#ffffff"))
            app.setPalette(pal)

        self.setStyleSheet("""
            QWidget { background: #0f1115; color: #e6e6e6; font-family: Segoe UI, Arial; }
            QToolTip { color: #e6e6e6; background-color: #0f1115; border: 1px solid #2b2f3a; }
            QFrame#ToolCard { background: #151a23; border: 1px solid #2b2f3a; border-radius: 12px; }
            QFrame#ToolCard:hover { border: 1px solid #2d6cdf; background: #171d28; }
            QLabel { color: #e6e6e6; }
            QPushButton { background: #1a2230; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 8px; padding: 6px 10px; font-weight: 600; }
            QPushButton:hover { background: #22304a; border: 1px solid #2d6cdf; }
            QPushButton:pressed { background: #2d6cdf; color: #ffffff; }
            QPushButton:disabled { background: #11151d; color: #7c7c7c; }
            QTextEdit { background: #12151c; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 8px; padding: 6px; }
            QScrollBar:vertical { background: #0f1115; width: 12px; margin: 0; }
            QScrollBar::handle:vertical { background: #2b2f3a; border-radius: 6px; min-height: 20px; }
        """)

    def on_launch_tool(self, tool: dict):
        try:
            proc = launch_tool(self.deps_dir, tool)
            self.processes.append(proc)
            self.status.setText(f"Launched: {tool.get('title', tool.get('id'))} | PID {proc.pid}")
            QMessageBox.information(self, "Tool launched", f"Started:\n{tool.get('title', tool.get('id'))}\n\nPID: {proc.pid}\n\nLogs are in:\n{self.deps_dir / 'logs'}")
        except Exception as e:
            append_log(self.deps_dir, f"FAILED launching {tool.get('id', '?')}: {e}\n{traceback.format_exc()}")
            QMessageBox.critical(self, "Launch failed", f"Could not launch tool:\n{tool.get('title', tool.get('id'))}\n\nError:\n{e}\n\nCheck logs in:\n{self.deps_dir / 'logs'}")

    def show_help(self):
        help_rel = self.config.get("help_file", "docs/PC_preparation_HELP.txt")
        help_path = self.deps_dir / help_rel
        if help_path.exists():
            help_text = help_path.read_text(encoding="utf-8")
        else:
            help_text = f"Help file not found:\n{help_path}"
        dlg = HelpDialog(help_text, self)
        dlg.exec()


def run_deepecohab_launcher():
    app = QApplication.instance()
    if app is None:
        app = QApplication([])

    deps = find_dependencies_folder()
    if deps is None:
        QMessageBox.warning(None, "Dependencies folder not found", f"Could not auto-detect '{DEPS_FOLDER_NAME}'.\n\nPlease select the DeepEcoHAB_dependencies folder.")
        selected = QFileDialog.getExistingDirectory(None, "Select DeepEcoHAB_dependencies folder")
        if not selected:
            raise RuntimeError("Dependencies folder not selected.")
        deps = Path(selected)
        if deps.name != DEPS_FOLDER_NAME and not (deps / CONFIG_FILENAME).exists():
            raise RuntimeError(f"Selected folder does not look like {DEPS_FOLDER_NAME}: {deps}")

    ensure_dirs(deps)
    cfg = load_config(deps)
    append_log(deps, f"Main menu opened. cwd={Path.cwd()} python={sys.executable}")
    global DEEPECOHAB_MAIN_MENU_WINDOW
    DEEPECOHAB_MAIN_MENU_WINDOW = MainMenuWindow(deps, cfg)
    DEEPECOHAB_MAIN_MENU_WINDOW.show()
    return DEEPECOHAB_MAIN_MENU_WINDOW

DEEPECOHAB_MAIN_MENU_WINDOW = run_deepecohab_launcher()


: 